# BigMart Data Preparation

This notebook handles the preparation stage for the BigMart project.

The goal is to load `Train.csv` and `Test.csv`, explore the training data, clean both datasets in a consistent way, apply the same feature engineering, and save prepared files for modeling later.

## 1. Imports

Only the libraries needed for preparation and basic exploration are used here.

In [ ]:
from pathlib import Path

import warnings
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
pd.set_option("display.max_columns", None)

## 2. Load Train and Test Data

Both files are loaded here, but the main exploration is done on the training set because that is the dataset with the target column.

In [ ]:
train_df = pd.read_csv("Train.csv")
test_df = pd.read_csv("Test.csv")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

In [ ]:
train_df.head()

## 3. Quick Exploration of the Training Data

This section keeps the exploration simple and focused on what matters before cleaning.

In [ ]:
train_df.info()

In [ ]:
train_missing = pd.DataFrame({
    "missing_count": train_df.isna().sum(),
    "missing_percent": (train_df.isna().mean() * 100).round(2)
}).sort_values("missing_count", ascending=False)

train_missing

In [ ]:
train_df.describe(include="all").T

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(train_df["Item_Outlet_Sales"], bins=40, kde=True)
plt.title("Distribution of Item Outlet Sales")
plt.xlabel("Item Outlet Sales")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

## 4. Cleaning and Feature Engineering Functions

The same preparation logic is used for both datasets.

For the test set, values are filled using information learned from the training data so that the process stays consistent.

In [ ]:
def clean_train_data(df: pd.DataFrame) -> pd.DataFrame:
    clean_df = df.copy()

    clean_df["Item_Fat_Content"] = clean_df["Item_Fat_Content"].replace({
        "LF": "Low Fat",
        "low fat": "Low Fat",
        "reg": "Regular"
    })

    clean_df["Item_Weight"] = clean_df["Item_Weight"].fillna(
        clean_df.groupby("Item_Identifier")["Item_Weight"].transform("mean")
    )
    clean_df["Item_Weight"] = clean_df["Item_Weight"].fillna(
        clean_df.groupby("Item_Type")["Item_Weight"].transform("mean")
    )

    outlet_size_map = clean_df.groupby("Outlet_Type")["Outlet_Size"].agg(lambda x: x.mode().iloc[0])
    clean_df["Outlet_Size"] = clean_df["Outlet_Size"].fillna(
        clean_df["Outlet_Type"].map(outlet_size_map)
    )

    item_visibility_mean = clean_df.groupby("Item_Identifier")["Item_Visibility"].transform("mean")
    overall_visibility_median = clean_df.loc[clean_df["Item_Visibility"] > 0, "Item_Visibility"].median()
    zero_visibility_mask = clean_df["Item_Visibility"] == 0
    replacement_visibility = item_visibility_mean.mask(item_visibility_mean == 0, overall_visibility_median)
    clean_df.loc[zero_visibility_mask, "Item_Visibility"] = replacement_visibility[zero_visibility_mask]

    clean_df["Item_Category"] = clean_df["Item_Identifier"].str[:2].map({
        "FD": "Food",
        "NC": "Non-Consumable",
        "DR": "Drinks"
    })
    clean_df.loc[clean_df["Item_Category"] == "Non-Consumable", "Item_Fat_Content"] = "Non-Edible"

    clean_df["Outlet_Age"] = 2013 - clean_df["Outlet_Establishment_Year"]
    clean_df["Item_Visibility_Mean"] = clean_df.groupby("Item_Identifier")["Item_Visibility"].transform("mean")
    clean_df["Visibility_Ratio"] = clean_df["Item_Visibility"] / clean_df["Item_Visibility_Mean"]

    return clean_df


def clean_test_data(df: pd.DataFrame, train_reference: pd.DataFrame) -> pd.DataFrame:
    clean_df = df.copy()
    reference_df = train_reference.copy()

    clean_df["Item_Fat_Content"] = clean_df["Item_Fat_Content"].replace({
        "LF": "Low Fat",
        "low fat": "Low Fat",
        "reg": "Regular"
    })

    item_weight_map = reference_df.groupby("Item_Identifier")["Item_Weight"].mean()
    item_type_weight_map = reference_df.groupby("Item_Type")["Item_Weight"].mean()
    outlet_size_map = reference_df.groupby("Outlet_Type")["Outlet_Size"].agg(lambda x: x.mode().iloc[0])
    item_visibility_map = reference_df.groupby("Item_Identifier")["Item_Visibility"].mean()
    overall_visibility_median = reference_df.loc[reference_df["Item_Visibility"] > 0, "Item_Visibility"].median()

    clean_df["Item_Weight"] = clean_df["Item_Weight"].fillna(clean_df["Item_Identifier"].map(item_weight_map))
    clean_df["Item_Weight"] = clean_df["Item_Weight"].fillna(clean_df["Item_Type"].map(item_type_weight_map))

    clean_df["Outlet_Size"] = clean_df["Outlet_Size"].fillna(clean_df["Outlet_Type"].map(outlet_size_map))

    zero_visibility_mask = clean_df["Item_Visibility"] == 0
    replacement_visibility = clean_df["Item_Identifier"].map(item_visibility_map).fillna(overall_visibility_median)
    clean_df.loc[zero_visibility_mask, "Item_Visibility"] = replacement_visibility[zero_visibility_mask]

    clean_df["Item_Category"] = clean_df["Item_Identifier"].str[:2].map({
        "FD": "Food",
        "NC": "Non-Consumable",
        "DR": "Drinks"
    })
    clean_df.loc[clean_df["Item_Category"] == "Non-Consumable", "Item_Fat_Content"] = "Non-Edible"

    clean_df["Outlet_Age"] = 2013 - clean_df["Outlet_Establishment_Year"]
    clean_df["Item_Visibility_Mean"] = clean_df["Item_Identifier"].map(item_visibility_map).fillna(overall_visibility_median)
    clean_df["Visibility_Ratio"] = clean_df["Item_Visibility"] / clean_df["Item_Visibility_Mean"]

    return clean_df

## 5. Apply the Preparation to Train and Test

The training data is cleaned first, and then the same logic is applied to the test data using the training set as reference.

In [ ]:
prepared_train = clean_train_data(train_df)
prepared_test = clean_test_data(test_df, prepared_train)

print("Prepared train shape:", prepared_train.shape)
print("Prepared test shape:", prepared_test.shape)

In [ ]:
print("Prepared train missing values:", int(prepared_train.isna().sum().sum()))
print("Prepared test missing values:", int(prepared_test.isna().sum().sum()))

In [ ]:
prepared_train.head()

## 6. Save the Prepared Files

These saved files will be used in the second notebook for modeling and final prediction.

In [ ]:
prepared_train.to_csv("prepared_train.csv", index=False)
prepared_test.to_csv("prepared_test.csv", index=False)

print("Saved prepared_train.csv")
print("Saved prepared_test.csv")

## 7. Stop Point

This notebook stops after preparation.

What stays in this notebook:
- loading `Train.csv` and `Test.csv`
- exploring mainly the training data
- cleaning both datasets consistently
- applying the same feature engineering to both
- saving `prepared_train.csv` and `prepared_test.csv`

What does not belong here:
- model training
- tuning
- prediction